# 加密货币 K 线分析

从 PostgreSQL 读取 OHLCV 数据，绘制交互式 K 线图 + 技术指标。

**数据来源**：`public.crypto_kline_binance`（1m K 线）  
**图表库**：Plotly（可缩放、悬停查看详情）

## ⚙️ 配置 — 在这里修改币种和日期区间

In [11]:
# ── 在这里修改参数 ──────────────────────────────────────────────
SYMBOL      = "BTC/USDT"                        # 数据库中的 symbol，例如 "ETHUSDT"
START_DATE  = "2026-04-09 00:00:00"             # 开始时间（本地时区），YYYY-MM-DD 或 YYYY-MM-DD HH:MM:SS
END_DATE    = "2026-04-12 00:00:00"             # 结束时间（本地时区，不含）
TIMEZONE    = "Asia/Shanghai"                  # 你所在的时区
DB_TABLE    = "public.crypto_kline_binance"    # 表名，1m K线
ONLY_CLOSED = True                             # True=只取已收盘的 K 线

# ── 爆仓数据 ────────────────────────────────────────────────────
SHOW_LIQUIDATIONS = True          # 是否在图表中叠加爆仓数据
# "binance_api" : 从 Binance FAPI 公开接口实时拉取（需联网，支持近 30 天）
# "database"    : 从数据库表读取（需自行采集入库）
LIQDATA_SOURCE    = "database"
LIQ_TABLE         = "public.crypto_liquidation_binance"  # 仅 source=database 时生效
# ────────────────────────────────────────────────────────────────

## 1. 导入 & 数据加载

In [12]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.db import read_ohlcv

# 将本地时间转换为 UTC 传给数据库
tz = TIMEZONE
start_utc = pd.Timestamp(START_DATE, tz=tz).tz_convert("UTC").isoformat()
end_utc   = pd.Timestamp(END_DATE,   tz=tz).tz_convert("UTC").isoformat()

# 从数据库读取（UTC 范围查询）
df = read_ohlcv(SYMBOL, start=start_utc, end=end_utc, table=DB_TABLE, only_closed=ONLY_CLOSED)

# 将 timestamp 转为本地时区显示（tz-naive 先标记为 UTC，再转换）
ts = df["timestamp"]
if ts.dt.tz is None:
    ts = ts.dt.tz_localize("UTC")
df["timestamp"] = ts.dt.tz_convert(tz)

print(f"Symbol      : {SYMBOL}")
print(f"Period      : {START_DATE} ~ {END_DATE}  ({tz})")
print(f"UTC range   : {start_utc}  →  {end_utc}")
print(f"Rows        : {len(df)}")
print(f"Time range  : {df['timestamp'].min()}  →  {df['timestamp'].max()}")
df.head()

Symbol      : BTC/USDT
Period      : 2026-04-09 00:00:00 ~ 2026-04-12 00:00:00  (Asia/Shanghai)
UTC range   : 2026-04-08T16:00:00+00:00  →  2026-04-11T16:00:00+00:00
Rows        : 3590
Time range  : 2026-04-09 12:09:00+08:00  →  2026-04-11 23:59:00+08:00


,timestamp,open,high,low,close,volume
0,2026-04-09 12:09:00+08:00,70940.00,70966.00,70939.99,70952.17,3.02934
1,2026-04-09 12:10:00+08:00,70952.17,70961.48,70952.16,70957.63,2.10522
2,2026-04-09 12:11:00+08:00,70957.63,70960.61,70954.64,70960.61,1.89865
3,2026-04-09 12:12:00+08:00,70960.60,70960.61,70925.27,70925.27,4.79019
4,2026-04-09 12:13:00+08:00,70925.28,70925.28,70888.47,70902.06,9.41687


## 1.2 爆仓数据加载

从 Binance FAPI 公开接口拉取强平（爆仓）订单，并聚合为 1 分钟粒度与 K 线对齐。

- **红色 (SELL)**：多单爆仓（强制卖出平多）
- **绿色 (BUY)** ：空单爆仓（强制买入平空）

In [13]:
import requests

def _fetch_liq_binance_api(symbol: str, start_ms: int, end_ms: int) -> pd.DataFrame:
    """从 Binance FAPI 公开接口拉取强平订单，自动分页（无需 API Key，支持近 30 天）。
    side='sell' → 多单被爆  |  side='buy' → 空单被爆
    cost = executedQty * averagePrice（实际成交金额，USD）
    """
    sym  = symbol.replace("/", "")
    url  = "https://fapi.binance.com/fapi/v1/allForceOrders"
    rows = []
    t    = start_ms
    while t < end_ms:
        params = dict(
            symbol    = sym,
            startTime = t,
            endTime   = min(t + 86_400_000, end_ms),
            limit     = 1000,
        )
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            t += 86_400_000
            continue
        rows.extend(batch)
        last_t = max(o["time"] for o in batch)
        t = last_t + 1 if len(batch) >= 1000 else last_t + 86_400_001
    if not rows:
        return pd.DataFrame(columns=["timestamp", "side", "qty", "avg_price", "cost"])
    df = (pd.DataFrame(rows)
          .drop_duplicates(subset=["time", "side", "price", "origQty"]))
    df["timestamp"] = pd.to_datetime(df["time"], unit="ms", utc=True)
    df["side"]      = df["side"].str.lower()
    df["qty"]       = df["executedQty"].astype(float)
    df["avg_price"] = df["averagePrice"].astype(float)
    df["cost"]      = df["qty"] * df["avg_price"]
    return df[["timestamp", "side", "qty", "avg_price", "cost"]].copy()


def _fetch_liq_database(symbol: str, start_utc: str, end_utc: str, table: str) -> pd.DataFrame:
    """从数据库表读取爆仓数据。
    表结构（public.crypto_liquidation_binance）：
      liquidation_time, symbol, side, price, avg_price, amount, filled, cost
    """
    from utils.db import execute
    sql = (
        f"SELECT liquidation_time AS timestamp, side, "
        f"       amount AS qty, avg_price, cost "
        f"FROM {table} "
        f"WHERE symbol = :sym "
        f"  AND liquidation_time >= :start "
        f"  AND liquidation_time <  :end "
        f"ORDER BY liquidation_time ASC"
    )
    df = execute(sql, {"sym": symbol, "start": start_utc, "end": end_utc})
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["side"]      = df["side"].str.lower()
    df[["qty", "avg_price", "cost"]] = df[["qty", "avg_price", "cost"]].astype(float)
    return df[["timestamp", "side", "qty", "avg_price", "cost"]].copy()


# ── 拉取爆仓数据 ──────────────────────────────────────────────────
start_ms = int(pd.Timestamp(START_DATE, tz=TIMEZONE).tz_convert("UTC").timestamp() * 1000)
end_ms   = int(pd.Timestamp(END_DATE,   tz=TIMEZONE).tz_convert("UTC").timestamp() * 1000)

if SHOW_LIQUIDATIONS:
    if LIQDATA_SOURCE == "binance_api":
        df_liq = _fetch_liq_binance_api(SYMBOL, start_ms, end_ms)
    else:
        df_liq = _fetch_liq_database(SYMBOL, start_utc, end_utc, LIQ_TABLE)
else:
    df_liq = pd.DataFrame(columns=["timestamp", "side", "qty", "avg_price", "cost"])

total_long  = df_liq.loc[df_liq["side"] == "sell", "cost"].sum()
total_short = df_liq.loc[df_liq["side"] == "buy",  "cost"].sum()
print(f"爆仓订单总数   : {len(df_liq):,}")
print(f"多单爆仓 (sell): {(df_liq['side']=='sell').sum():,} 笔   {total_long/1e6:.3f} M USD")
print(f"空单爆仓 (buy) : {(df_liq['side']=='buy').sum():,} 笔   {total_short/1e6:.3f} M USD")
df_liq.head()


爆仓订单总数   : 1,077
多单爆仓 (sell): 383 笔   4.372 M USD
空单爆仓 (buy) : 694 笔   8.529 M USD


,timestamp,side,qty,avg_price,cost
0,2026-04-10 01:57:50.479000+00:00,sell,0.005,72110.4,360.5520
1,2026-04-10 02:04:37.263000+00:00,sell,0.002,72085.3,144.1706
2,2026-04-10 02:23:22.563000+00:00,buy,0.024,72163.8,1731.9312
3,2026-04-10 02:24:31.130000+00:00,buy,0.010,72170.0,721.7000
4,2026-04-10 02:45:45.140000+00:00,sell,0.013,72060.4,936.7852


In [14]:
def _resample_liq(df: pd.DataFrame, tz: str) -> pd.DataFrame:
    """将爆仓订单按1分钟（K线 open_time 对齐）聚合为多/空爆仓金额（USD cost）。"""
    long_idx  = df.loc[df["side"] == "sell"].set_index("timestamp")["cost"]
    short_idx = df.loc[df["side"] == "buy"].set_index("timestamp")["cost"]
    liq_long  = (long_idx .resample("1min", label="left", closed="left")
                 .sum().rename("liq_long"))
    liq_short = (short_idx.resample("1min", label="left", closed="left")
                 .sum().rename("liq_short"))
    agg = pd.concat([liq_long, liq_short], axis=1).fillna(0)
    agg.index = agg.index.tz_convert(tz)
    return agg.reset_index()

liq_1m = _resample_liq(df_liq, TIMEZONE)
print(f"1分钟爆仓汇总行数     : {len(liq_1m)}")
print(f"最大单分钟多单爆仓量  : {liq_1m['liq_long'].max()/1e3:.2f} K USD")
print(f"最大单分钟空单爆仓量  : {liq_1m['liq_short'].max()/1e3:.2f} K USD")
liq_1m[liq_1m["liq_long"] + liq_1m["liq_short"] > 0].head(8)


1分钟爆仓汇总行数     : 2283
最大单分钟多单爆仓量  : 1028.01 K USD
最大单分钟空单爆仓量  : 2141.74 K USD


,timestamp,liq_long,liq_short
0,2026-04-10 09:57:00+08:00,360.5520,0.0000
7,2026-04-10 10:04:00+08:00,144.1706,0.0000
26,2026-04-10 10:23:00+08:00,0.0000,1731.9312
27,2026-04-10 10:24:00+08:00,0.0000,721.7000
48,2026-04-10 10:45:00+08:00,6701.9772,0.0000
49,2026-04-10 10:46:00+08:00,864.5526,0.0000
50,2026-04-10 10:47:00+08:00,936.2449,0.0000
52,2026-04-10 10:49:00+08:00,7629.0324,0.0000


## 2. 数据质检

In [15]:
print("=== 缺失值 ===")
print(df.isnull().sum())

print("\n=== 基本统计 ===")
print(df[["open", "high", "low", "close", "volume"]].describe().round(4))

# 检查 high >= low 和 high >= close
anomalies = df[(df["high"] < df["low"]) | (df["high"] < df["close"]) | (df["low"] > df["close"])]
print(f"\n异常 OHLC 行数: {len(anomalies)}")
if not anomalies.empty:
    print(anomalies)

=== 缺失值 ===
timestamp    0
open         0
high         0
low          0
close        0
volume       0
dtype: int64

=== 基本统计 ===
             open        high         low       close     volume
count   3590.0000   3590.0000   3590.0000   3590.0000  3590.0000
mean   72213.1433  72232.3084  72194.6148  72213.6617    10.4227
std      696.6137    694.9078    698.2708    696.3369    18.7831
min    70560.0700  70598.2400  70522.7700  70560.0800     0.1051
25%    71778.6000  71797.8200  71758.6125  71780.4275     2.5574
50%    72272.1250  72299.4400  72246.4100  72272.3450     5.3041
75%    72829.8000  72838.8300  72817.1925  72829.8000    10.7535
max    73412.2500  73434.0000  73355.5900  73412.2500   517.6822

异常 OHLC 行数: 0


## 3. 技术指标计算

In [16]:
d = df.copy()

# 均线
d["ema9"]  = d["close"].ewm(span=9,  adjust=False).mean()
d["ema21"] = d["close"].ewm(span=21, adjust=False).mean()
d["ema55"] = d["close"].ewm(span=55, adjust=False).mean()

# 布林带 (20, 2σ)
d["bb_mid"]   = d["close"].rolling(20).mean()
d["bb_std"]   = d["close"].rolling(20).std()
d["bb_upper"] = d["bb_mid"] + 2 * d["bb_std"]
d["bb_lower"] = d["bb_mid"] - 2 * d["bb_std"]

# RSI (14)
delta = d["close"].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
d["rsi"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))

# MACD (12, 26, 9)
ema12 = d["close"].ewm(span=12, adjust=False).mean()
ema26 = d["close"].ewm(span=26, adjust=False).mean()
d["macd"]        = ema12 - ema26
d["macd_signal"] = d["macd"].ewm(span=9, adjust=False).mean()
d["macd_hist"]   = d["macd"] - d["macd_signal"]

# 成交量均线
d["vol_ma5"]  = d["volume"].rolling(5).mean()
d["vol_ma20"] = d["volume"].rolling(20).mean()

d = d.dropna().reset_index(drop=True)
print(f"计算完成，有效行数: {len(d)}")

计算完成，有效行数: 3571


## 4. K 线图（含均线 + 布林带 + 成交量 + 爆仓 + RSI + MACD）

In [17]:
# ── 合并爆仓数据到 K 线 ─────────────────────────────────────────
# 用 floor('min') 做连接键，消除 K 线 open_time 的亚秒精度差异
_d_key   = d.assign(_mk=d["timestamp"].dt.floor("min"))
_liq_key = liq_1m.assign(_mk=liq_1m["timestamp"].dt.floor("min"))
d_plot = _d_key.merge(
    _liq_key[["_mk", "liq_long", "liq_short"]],
    on="_mk", how="left"
).drop(columns=["_mk"])
d_plot[["liq_long", "liq_short"]] = d_plot[["liq_long", "liq_short"]].fillna(0)
print(f"爆仓非零分钟数: {(d_plot['liq_long']+d_plot['liq_short']>0).sum()} / {len(d_plot)}")

t = d_plot["timestamp"]

fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    row_heights=[0.50, 0.12, 0.13, 0.12, 0.13],
    vertical_spacing=0.02,
    subplot_titles=(
        f"{SYMBOL}  K线",
        "成交量",
        "爆仓量 (K USD)  |  红=多单爆  绿=空单爆",
        "RSI (14)",
        "MACD",
    ),
)

# ── Row 1: K 线 ──────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=t, open=d_plot["open"], high=d_plot["high"],
    low=d_plot["low"], close=d_plot["close"],
    name="K线",
    increasing_line_color="#ef5350", decreasing_line_color="#26a69a",
    increasing_fillcolor="#ef5350",  decreasing_fillcolor="#26a69a",
), row=1, col=1)

for col_name, color, name in [
    ("ema9",  "#ffeb3b", "EMA9"),
    ("ema21", "#ff9800", "EMA21"),
    ("ema55", "#ce93d8", "EMA55"),
]:
    fig.add_trace(go.Scatter(
        x=t, y=d_plot[col_name],
        line=dict(color=color, width=1), name=name,
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=t, y=d_plot["bb_upper"],
    line=dict(color="rgba(100,181,246,0.6)", width=1, dash="dot"), name="BB上轨",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=t, y=d_plot["bb_lower"],
    line=dict(color="rgba(100,181,246,0.6)", width=1, dash="dot"), name="BB下轨",
    fill="tonexty", fillcolor="rgba(100,181,246,0.05)",
), row=1, col=1)

# ── Row 2: 成交量 ────────────────────────────────────────────────
vol_colors = ["#ef5350" if c >= o else "#26a69a"
              for c, o in zip(d_plot["close"], d_plot["open"])]
fig.add_trace(go.Bar(
    x=t, y=d_plot["volume"],
    marker_color=vol_colors, name="成交量", showlegend=False,
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=t, y=d_plot["vol_ma5"],
    line=dict(color="#ffeb3b", width=1), name="VOL MA5",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=t, y=d_plot["vol_ma20"],
    line=dict(color="#ff9800", width=1), name="VOL MA20",
), row=2, col=1)

# ── Row 3: 爆仓量 ────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=t, y=d_plot["liq_long"] / 1e3,
    marker_color="rgba(239,83,80,0.85)",
    name="多单爆仓 (K USD)",
), row=3, col=1)
fig.add_trace(go.Bar(
    x=t, y=d_plot["liq_short"] / 1e3,
    marker_color="rgba(38,166,154,0.85)",
    name="空单爆仓 (K USD)",
), row=3, col=1)

# ── Row 4: RSI ───────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=t, y=d_plot["rsi"],
    line=dict(color="#ba68c8", width=1.5), name="RSI",
), row=4, col=1)
fig.add_hline(y=70, line=dict(color="red",   width=1, dash="dash"), row=4, col=1)
fig.add_hline(y=30, line=dict(color="green", width=1, dash="dash"), row=4, col=1)
fig.add_hrect(y0=30, y1=70, fillcolor="rgba(255,255,255,0.03)", line_width=0, row=4, col=1)

# ── Row 5: MACD ──────────────────────────────────────────────────
hist_colors = ["#ef5350" if v >= 0 else "#26a69a" for v in d_plot["macd_hist"]]
fig.add_trace(go.Bar(
    x=t, y=d_plot["macd_hist"],
    marker_color=hist_colors, name="MACD Hist", showlegend=False,
), row=5, col=1)
fig.add_trace(go.Scatter(
    x=t, y=d_plot["macd"],
    line=dict(color="#2196f3", width=1.5), name="MACD",
), row=5, col=1)
fig.add_trace(go.Scatter(
    x=t, y=d_plot["macd_signal"],
    line=dict(color="#ff9800", width=1.5), name="Signal",
), row=5, col=1)

# ── 布局 ─────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=f"{SYMBOL}  {START_DATE} ~ {END_DATE}  |  爆仓: 红=多单 / 绿=空单",
        font=dict(size=16),
    ),
    height=1050,
    template="plotly_dark",
    xaxis_rangeslider_visible=False,
    barmode="overlay",
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
    margin=dict(l=60, r=20, t=80, b=20),
    hovermode="x unified",
)
fig.update_yaxes(title_text="价格",        row=1, col=1)
fig.update_yaxes(title_text="成交量",      row=2, col=1)
fig.update_yaxes(title_text="爆仓 (KUSD)", row=3, col=1)
fig.update_yaxes(title_text="RSI",         row=4, col=1, range=[0, 100])
fig.update_yaxes(title_text="MACD",        row=5, col=1)

fig.show()

爆仓非零分钟数: 487 / 3571


## 4.1 爆仓与行情关联分析

- **Row 1**：多/空单爆仓量（K USD）柱状图
- **Row 2**：归一化对比 — 爆仓量（橙）vs 绝对价格变动（蓝）；两者峰值重合说明爆仓与剧烈波动同步
- **Row 3**：滚动 60m Pearson 相关系数；穿越 ±0.5 虚线为显著相关

In [18]:
def _norm(s: pd.Series) -> pd.Series:
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn + 1e-10)

# ── 构建分析表 ────────────────────────────────────────────────────
an = d_plot[["timestamp", "open", "close", "volume", "liq_long", "liq_short"]].copy()
an["price_change"]  = an["close"].diff().abs()          # 绝对价格变动
an["pct_change"]    = an["close"].pct_change().abs()    # 绝对收益率
an["total_liq"]     = an["liq_long"] + an["liq_short"]
an["liq_net"]       = an["liq_long"] - an["liq_short"]  # >0 多单主导，<0 空单主导
an["total_liq_log"] = np.log1p(an["total_liq"])

# 滚动相关系数（60分钟窗口）
WIN = 60
an["rolling_corr"] = (
    an["total_liq_log"].rolling(WIN).corr(an["price_change"])
)

# ── 整体相关系数（仅有爆仓记录的分钟）────────────────────────────
mask = an["total_liq"] > 0
print(f"有爆仓记录的分钟数 : {mask.sum()} / {len(an)}")
print(f"\n=== Pearson 相关系数（爆仓量 log vs ...）===")
subset = an.loc[mask, ["total_liq_log", "price_change", "pct_change", "volume"]]
corr_row = subset.corr()["total_liq_log"].drop("total_liq_log")
for k, v in corr_row.items():
    print(f"  vs {k:<18}: {v:+.4f}")

# ── 绘图 ─────────────────────────────────────────────────────────
fig_liq = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.38, 0.30, 0.32],
    vertical_spacing=0.04,
    subplot_titles=(
        "爆仓量 (K USD)  |  多单爆(红) / 空单爆(绿)",
        f"归一化对比：总爆仓量 vs 绝对价格变动",
        f"滚动 {WIN}m 相关系数（总爆仓量 log vs 绝对价格变动）",
    ),
)

t = an["timestamp"]

# Row 1: 多/空爆仓柱
fig_liq.add_trace(go.Bar(
    x=t, y=an["liq_long"] / 1e3,
    marker_color="rgba(239,83,80,0.85)", name="多单爆仓",
), row=1, col=1)
fig_liq.add_trace(go.Bar(
    x=t, y=an["liq_short"] / 1e3,
    marker_color="rgba(38,166,154,0.85)", name="空单爆仓",
), row=1, col=1)

# Row 2: 归一化对比（同轴）
fig_liq.add_trace(go.Scatter(
    x=t, y=_norm(an["total_liq"]),
    line=dict(color="#ffa726", width=1.2), name="总爆仓量(归一化)",
), row=2, col=1)
fig_liq.add_trace(go.Scatter(
    x=t, y=_norm(an["price_change"]),
    line=dict(color="#64b5f6", width=1.2), name="绝对价格变动(归一化)",
), row=2, col=1)

# Row 3: 滚动相关系数
fig_liq.add_trace(go.Scatter(
    x=t, y=an["rolling_corr"],
    line=dict(color="#ce93d8", width=1.5), name=f"滚动{WIN}m 相关系数",
    fill="tozeroy", fillcolor="rgba(206,147,216,0.12)",
), row=3, col=1)
fig_liq.add_hline(y=0,    line=dict(color="white",   width=1, dash="dot"),  row=3, col=1)
fig_liq.add_hline(y=0.5,  line=dict(color="#ef5350", width=1, dash="dash"), row=3, col=1)
fig_liq.add_hline(y=-0.5, line=dict(color="#26a69a", width=1, dash="dash"), row=3, col=1)

fig_liq.update_layout(
    title=dict(
        text=f"{SYMBOL}  爆仓 × 行情 关联分析  |  {START_DATE} ~ {END_DATE}",
        font=dict(size=15),
    ),
    height=620,
    template="plotly_dark",
    barmode="overlay",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
    margin=dict(l=60, r=20, t=80, b=20),
)
fig_liq.update_yaxes(title_text="爆仓 (KUSD)", row=1, col=1)
fig_liq.update_yaxes(title_text="归一化",      row=2, col=1, range=[0, 1])
fig_liq.update_yaxes(title_text="相关系数",    row=3, col=1, range=[-1, 1])

fig_liq.show()

有爆仓记录的分钟数 : 487 / 3571

=== Pearson 相关系数（爆仓量 log vs ...）===
  vs price_change      : +0.2916
  vs pct_change        : +0.2915
  vs volume            : +0.3381


## 5. 收益率分布

In [19]:
ret = d["close"].pct_change().dropna()
log_ret = np.log(d["close"] / d["close"].shift(1)).dropna()

fig2 = make_subplots(rows=1, cols=2, subplot_titles=("简单收益率分布", "对数收益率分布"))

for col_idx, (r, name) in enumerate([(ret, "简单收益率"), (log_ret, "对数收益率")], start=1):
    fig2.add_trace(go.Histogram(
        x=r, nbinsx=80,
        marker_color="rgba(100,181,246,0.7)",
        name=name, showlegend=False,
    ), row=1, col=col_idx)
    # 零线
    fig2.add_vline(x=0, line=dict(color="red", dash="dash", width=1), row=1, col=col_idx)

ann = (
    f"均值: {ret.mean():.4f}<br>"
    f"标准差: {ret.std():.4f}<br>"
    f"偏度: {ret.skew():.4f}<br>"
    f"峰度: {ret.kurt():.4f}<br>"
    f"年化收益: {(1 + ret.mean()) ** 365 - 1:.2%}<br>"
    f"年化波动: {ret.std() * np.sqrt(365):.2%}<br>"
    f"年化夏普: {ret.mean() / ret.std() * np.sqrt(365):.2f}"
)
fig2.add_annotation(
    xref="paper", yref="paper", x=1.01, y=0.95,
    text=ann, showarrow=False, align="left",
    font=dict(size=12, color="white"),
    bordercolor="gray", borderwidth=1, bgcolor="rgba(0,0,0,0.5)",
)

fig2.update_layout(
    title=f"{SYMBOL} 收益率分布  {START_DATE} ~ {END_DATE}",
    height=420, template="plotly_dark",
    margin=dict(l=60, r=180, t=60, b=40),
)
fig2.show()

## 6. 关键统计摘要

In [20]:
price_chg = (d["close"].iloc[-1] / d["close"].iloc[0] - 1) * 100
high_price = d["high"].max()
low_price  = d["low"].min()
drawdown   = (d["close"] / d["close"].cummax() - 1)
max_dd     = drawdown.min() * 100
avg_vol    = d["volume"].mean()

print(f"{'═' * 40}")
print(f"  {SYMBOL}  {START_DATE} ~ {END_DATE}")
print(f"{'═' * 40}")
print(f"  期间涨跌幅   : {price_chg:+.2f}%")
print(f"  最高价       : {high_price:,.2f}")
print(f"  最低价       : {low_price:,.2f}")
print(f"  最大回撤     : {max_dd:.2f}%")
print(f"  年化收益     : {(1 + ret.mean()) ** 365 - 1:.2%}")
print(f"  年化波动率   : {ret.std() * np.sqrt(365):.2%}")
print(f"  年化夏普比率 : {ret.mean() / ret.std() * np.sqrt(365):.2f}")
print(f"  平均成交量   : {avg_vol:,.0f}")
print(f"{'═' * 40}")

════════════════════════════════════════
  BTC/USDT  2026-04-09 00:00:00 ~ 2026-04-12 00:00:00
════════════════════════════════════════
  期间涨跌幅   : +2.86%
  最高价       : 73,434.00
  最低价       : 70,522.77
  最大回撤     : -2.13%
  年化收益     : 0.29%
  年化波动率   : 1.01%
  年化夏普比率 : 0.29
  平均成交量   : 10
════════════════════════════════════════
